# Agents and the Function Kernel

This notebook introduces the two building blocks behind Kaval.AI's tool-using
agents:

1. The **`FunctionKernel`** — a single, uniform interface for calling Python,
   REST and MCP tools, with automatic Pydantic validation of every input and
   output.
2. The **`Agent`** — a lightweight multi-step reasoning loop that plans, calls
   those tools, and reasons over the results to produce a typed answer.

We first build up the kernel from plain Python functions, then plug it into an
agent and work through four worked examples (Python tools, web scraping, REST
and MCP).

## Function kernel

The `FunctionKernel` is the component that lets an agent call tools. It hosts three kinds of tools behind a single, uniform interface:

- **Python functions** — addressed as `python://<name>`
- **REST endpoints** — addressed as `rest://<server>.<tool>`
- **MCP tools** — addressed as `mcp://<server>.<tool>`

Every tool has an **input model** and an **output model**, both Pydantic `BaseModel` subclasses. The kernel validates and coerces arguments against the input model before a call, and coerces the return value into the output model afterwards. As a result an agent always sees well-typed, validated data, regardless of how the underlying tool was written.

The sections below build up two filesystem tools — `ls` and `read_file` — and show exactly how plain Python function signatures become these Pydantic models.

### Defining Python tools with `@pythontool`

A Python tool is just a function decorated with `@pythontool`. The decorator marks the function as a kavalai tool (it sets an internal `_is_kavalai_tool` flag) — it does **not** change behaviour, so the function stays directly callable in tests and other code.

To make a tool callable by an agent, register it on a kernel:

```python
kernel.register_python_tool("fs.ls", ls)
```

The first argument is the tool's **name**; the kernel then exposes it under the URI `python://fs.ls`. Names are free-form strings — dotted names like `fs.ls` are simply a convention for grouping related tools.

Write tools with ordinary type hints and a docstring. As the next sections show, the hints become the input/output Pydantic models and the docstring becomes the tool description the agent sees.

In [1]:
import os
from kavalai.functionkernel import FunctionKernel, pythontool

# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))


@pythontool
def ls(path: str = ".") -> list[str]:
    """List files and directories at `path`, relative to the project root."""
    return sorted(os.listdir(os.path.join(PROJECT_ROOT, path)))


@pythontool
def read_file(path: str, max_chars: int = 4000) -> str:
    """Read a UTF-8 text file at `path` (relative to the project root), returning up to `max_chars` characters."""
    with open(os.path.join(PROJECT_ROOT, path), "r", encoding="utf-8") as f:
        return f.read()[:max_chars]


# A `@pythontool` function is unchanged — we can still call it directly.
print(ls("kavalai")[:15])

# Instantiate a function kernel and register both tools under `python://` URIs.
kernel = FunctionKernel()
kernel.register_python_tool("fs.ls", ls)
kernel.register_python_tool("fs.read_file", read_file)

/home/timo/projects/kaval.ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['__init__.py', '__pycache__', 'agents', 'backoffice', 'crud.py', 'demo_agents', 'functionkernel.py', 'llm_clients', 'migrate_db.py', 'normalizer.py', 'paths.py', 'persona_simulator.py', 'sql_migrations', 'tools']


### How inputs map to Pydantic models

The **input model** is generated from the function signature:

| Function parameter | Pydantic field |
|---|---|
| parameter name | field name |
| type hint (e.g. `str`, `int`, `list[str]`) | field type |
| default value (`path="."`) | field default (field is *optional*) |
| no default (`path`) | field is *required* |
| no type hint | field typed as `Any` |

When a tool is called, the kernel instantiates this model with the supplied arguments, so values are validated and coerced to the declared types (for example the string `"50"` becomes the int `50`).

In [2]:
# `ls(path=".")` -> one optional field; `read_file(path, max_chars=4000)` -> one
# required and one optional field.
ls_input = kernel.get_input_model("python://fs.ls")
read_file_input = kernel.get_input_model("python://fs.read_file")

print("fs.ls input fields:       ", ls_input.model_fields)
print("fs.read_file input fields:", read_file_input.model_fields)

# Arguments are coerced to the declared types: the string "50" becomes int 50.
print("coerced:", read_file_input(path="README.md", max_chars="50"))

fs.ls input fields:        {'path': FieldInfo(annotation=str, required=False, default='.')}
fs.read_file input fields: {'path': FieldInfo(annotation=str, required=True), 'max_chars': FieldInfo(annotation=int, required=False, default=4000)}
coerced: path='README.md' max_chars=50


### How outputs map to Pydantic models

The **output model** is derived from the return annotation:

- If the function already returns a Pydantic `BaseModel`, that model is used as-is.
- Otherwise the kernel wraps the return value in a generated model with a single field named **`result`** of the annotated type. So `ls() -> list[str]` produces a model with `result: list[str]`, and `read_file() -> str` produces `result: str`.

`call_tool` runs the validated call and returns an **instance** of this output model — read the plain value back from `.result` (or directly from the model's fields when the tool returns a `BaseModel`).

In [3]:
# Both plain-return tools wrap their value in a `result` field.
print("fs.ls output fields:       ", kernel.get_output_model("python://fs.ls").model_fields)
print("fs.read_file output fields:", kernel.get_output_model("python://fs.read_file").model_fields)

# call_tool validates the arguments, runs the function and returns the output model.
listing = await kernel.call_tool("python://fs.ls")
print("\nproject root:", listing.result)

readme = await kernel.call_tool(
    "python://fs.read_file", arguments={"path": "README.md", "max_chars": 200}
)
print("\nREADME.md (first 200 chars):\n", readme.result)

fs.ls output fields:        {'result': FieldInfo(annotation=list[str], required=True)}
fs.read_file output fields: {'result': FieldInfo(annotation=str, required=True)}

project root: ['.claude', '.dockerignore', '.env', '.env.example', '.git', '.gitignore', '.idea', '.junie', '.pre-commit-config.yaml', '.prettierrc', '.pytest_cache', '.ruff_cache', '.run', '.trunk', '.venv', 'CLAUDE.md', 'LICENSE.txt', 'README.md', '__pycache__', 'build', 'docker-compose.yml', 'dockerfiles', 'docs', 'examples', 'frontend', 'junie.md', 'kavalai', 'kavalai.egg-info', 'local_data', 'notebooks', 'pyproject.toml', 'scripts', 'test_results.log', 'test_results_2.log', 'tests', 'tslint.json', 'uv.lock', 'workflows']

README.md (first 200 chars):
 <img src="frontend/public/assets/images/iconlogo.svg" alt="Kaval.AI Logo" width="400" height="100"/>

Kaval.AI is an open source Python SDK for writing AI agents and workflow automation pipelines.
Sui


## Agents

`kavalai.agents.agent.Agent` is a lightweight multi-step reasoning loop that:

1. Renders a system prompt from a Jinja2 template with the task, available tools, and step history.
2. Calls the LLM to get a `StepOutput` — a list of tool calls **plus** an optional final answer.
3. Executes tool calls in parallel through the `FunctionKernel` and feeds results back into the next step.
4. Stops when the model returns an answer without further tool calls or `max_steps` is reached.

Any v2 LLM client (`OpenAIClient`, `GeminiClient`, `OllamaClient`) can be plugged in.

In [4]:
# Load environment variables from the project .env file
from dotenv import load_dotenv
load_dotenv()

from kavalai.agents.agent import Agent
from kavalai.agents.run_context import RunContext
from kavalai.agents.workflow_model import RestServer, McpServer
from kavalai.functionkernel import FunctionKernel, pythontool
from kavalai.llm_clients.gemini_client import GeminiClient
from kavalai.llm_clients.openai_client import OpenAIClient

### Example 1 — Project Summarizer Agent

The agent reuses the two filesystem tools defined in the function-kernel section above and reasons over what it finds to fill in a structured `ProjectSummary`:
- `python://fs.ls` — list a directory, relative to the project root
- `python://fs.read_file` — read a text file, relative to the project root

A typical run lists the project root, reads informative files (`README.md`, `CLAUDE.md`, `pyproject.toml`, a few key source files), then synthesises a summary of the Kaval.AI project's purpose, tech stack and main components.

In [5]:
from pydantic import BaseModel, Field, ConfigDict


class ProjectSummary(BaseModel):
    """Structured summary of a software project discovered on disk."""

    model_config = ConfigDict(extra="forbid")

    name: str = Field(description="The name of the project.")
    summary: str = Field(description="A concise paragraph describing what the project does.")
    tech_stack: list[str] = Field(
        default_factory=list,
        description="Key languages, frameworks and tools the project uses.",
    )
    key_components: list[str] = Field(
        default_factory=list,
        description="Main components or modules and a short note on each one's role.",
    )

In [6]:
import json

# Reuse the fs.ls / fs.read_file tools defined in the function kernel section above.
kernel = FunctionKernel()
kernel.register_python_tool("fs.ls", ls)
kernel.register_python_tool("fs.read_file", read_file)

# v2 LLM client — swap to OpenAIClient("gpt-4.1-mini") if preferred
llm_client = GeminiClient("gemini-3.1-flash-lite")

agent = Agent(llm_client=llm_client, kernel=kernel)

result = await agent.prompt(
    prompt=(
        "Summarize the software project in the project root. "
        "Begin by listing the root directory with fs.ls, then read the most "
        "informative files (such as README.md, CLAUDE.md, pyproject.toml and a few "
        "key source files) with fs.read_file to understand the project's purpose, "
        "tech stack and main components. All paths are relative to the project root."
    ),
    response_model=ProjectSummary,
    max_steps=8,
)

print(json.dumps(result.model_dump(), indent=2))

2026-06-15 16:44:39.008 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 0/8
2026-06-15 16:44:39.796 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-3.1-flash-lite): 782 tokens, 0.78s
2026-06-15 16:44:39.797 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://fs.ls with {'path': '.'}
2026-06-15 16:44:39.804 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 1/8
2026-06-15 16:44:40.902 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-3.1-flash-lite): 1106 tokens, 1.10s
2026-06-15 16:44:40.903 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://fs.read_file with {'path': 'README.md'}
2026-06-15 16:44:40.905 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://fs.read_file with {'path': 'CLAUDE.md'}
2026-06-15 16:44:40.906 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://fs.read_file with {'pa

{
  "name": "Kaval.AI",
  "summary": "Kaval.AI is an open-source enterprise agent management system and Python SDK designed for building, configuring, and monitoring AI agents and workflow automation pipelines. It allows users to define complex workflows using YAML, supports multi-step planning agents with tool-calling capabilities (REST, MCP, Python), and provides a built-in backoffice UI for management.",
  "tech_stack": [
    "Python 3.12+",
    "FastAPI",
    "SQLAlchemy (Async)",
    "Pydantic",
    "Angular",
    "Tailwind CSS",
    "DaisyUI",
    "PostgreSQL",
    "uv"
  ],
  "key_components": [
    "kavalai.agents: Core SDK containing the workflow engine, planning agent, and session management.",
    "kavalai.backoffice: FastAPI-based management API for agent configuration and monitoring.",
    "frontend: Angular-based UI for interacting with the backoffice.",
    "kavalai.llm_clients: Abstraction layer for various LLM providers (OpenAI, Gemini, Ollama).",
    "kavalai.function

### Example 2 — Business Info Agent (Crawl4AI)

This agent scrapes a company website to fill in a structured `BusinessInfo` model. It uses the **Crawl4AI** web tool that ships with Kaval.AI — `kavalai.tools.webtools.crawl4ai.crawl_url`, registered here as `python://web.crawl`. Crawl4AI renders the page in a headless browser (JavaScript included) and returns clean **Markdown** instead of raw HTML, which is far easier for the LLM to read.

Because the Markdown keeps links inline (as `[text](url)`), a single tool is enough: the agent crawls the home page, follows links to relevant subpages (About, Contact, Imprint, …), crawls the most promising ones, and reasons over the combined content to fill in the fields.

> Requires the optional `tools` dependency (`pip install "kavalai[tools]"`) and a Playwright browser (`crawl4ai-setup`). No search/scrape API key is needed.

In [13]:
from typing import Optional

from pydantic import BaseModel, Field, ConfigDict

# The Crawl4AI web tool shipped with Kaval.AI — already decorated with @pythontool.
from kavalai.tools.webtools.crawl4ai import crawl_url


class BusinessInfo(BaseModel):
    """Structured information about a business found online."""

    model_config = ConfigDict(extra="forbid")

    name: str = Field(description="The legal or trading name of the business.")
    vat_number: Optional[str] = Field(None, description="The VAT number of the business.")
    website: Optional[str] = Field(None, description="The official website URL.")
    description: str = Field(description="Brief description of what the business does.")
    industry: Optional[str] = Field(None, description="The industry it operates in.")


# `crawl_url(url, include_html=False, bypass_cache=False, timeout=60.0)` renders the
# page in a headless browser and returns a `Crawl4aiResponse` whose `markdown` field
# holds clean Markdown (with inline links the agent can follow).
print(crawl_url.__doc__)


    Crawl a web page and return its content as clean Markdown using Crawl4AI.

    Args:
        url: The website URL to crawl.
        include_html: Whether to also return the cleaned HTML (default False).
        bypass_cache: Whether to bypass the crawler cache and fetch fresh content (default False).
        timeout: Page load timeout in seconds (default 60.0).
    


In [17]:
import json

# Build a kernel with the Crawl4AI web tool — no API key needed.
kernel = FunctionKernel()
kernel.register_python_tool("web.crawl", crawl_url)

llm_client = OpenAIClient("gpt-5.4")

agent = Agent(llm_client=llm_client, kernel=kernel)

result = await agent.prompt(
    prompt=(
        "Find information about the company 'Kaval AI'. Its website is https://kaval.ai/.\n"
        "1. Crawl the home page with web.crawl to get its Markdown content.\n"
        "2. The Markdown keeps links inline as [text](url) — follow the ones pointing "
        "to relevant subpages such as About, Contact or Imprint and crawl them too "
        "with web.crawl.\n"
        "3. Reason over the collected content to fill in all fields of the business profile."
    ),
    response_model=BusinessInfo,
    max_steps=8,
)

print(json.dumps(result.model_dump(), indent=2))

2026-06-15 17:51:16.156 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 0/8
2026-06-15 17:51:18.520 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-5.4): 1510 tokens, 2.36s
2026-06-15 17:51:18.521 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://web.crawl with {'url': 'https://kaval.ai/', 'include_html': False, 'bypass_cache': False, 'timeout': 60.0}
2026-06-15 17:51:18.579 | INFO     | kavalai.tools.webtools.crawl4ai:crawl_url:60 - Crawling URL with Crawl4AI: https://kaval.ai/


[INIT].... → Crawl4AI 0.8.9 

[FETCH]... ↓ https://kaval.ai/                                                                                    | ✓ | ⏱: 0.00s 

[COMPLETE] ● https://kaval.ai/                                                                                    | ✓ | ⏱: 0.01s 

2026-06-15 17:51:19.200 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 1/8
2026-06-15 17:51:21.514 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-5.4): 1842 tokens, 2.31s
2026-06-15 17:51:21.515 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://web.crawl with {'url': 'https://kaval.ai/contact', 'include_html': False, 'bypass_cache': False, 'timeout': 60.0}
2026-06-15 17:51:21.622 | INFO     | kavalai.tools.webtools.crawl4ai:crawl_url:60 - Crawling URL with Crawl4AI: https://kaval.ai/contact
2026-06-15 17:51:21.624 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool python://web.crawl with {'url': 'https://kaval.ai/terms', 'include_html': False, 'bypass_cache': False, 'timeout': 60.0}
2026-06-15 17:51:21.690 | INFO     | kavalai.tools.webtools.crawl4ai:crawl_url:60 - Crawling URL with Crawl4AI: https://kaval.ai/terms
2026-06-15 17:51:21.693 | INFO     | kavalai.agents.agent:_call_tool:241 - Call

...

[FETCH]... ↓ https://kaval.ai/privacy                                                                             | ✓ | ⏱: 0.97s 

[SCRAPE].. ◆ https://kaval.ai/privacy                                                                             | ✓ | ⏱: 0.01s 

[COMPLETE] ● https://kaval.ai/privacy                                                                             | ✓ | ⏱: 0.99s 

2026-06-15 17:51:23.496 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 2/8
2026-06-15 17:51:25.459 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-5.4): 3796 tokens, 1.96s


{
  "name": "Kaval.AI",
  "vat_number": "EE102934472",
  "website": "https://kaval.ai/",
  "description": "AI consultancy and development services company focused on AI agents and agent development toolkit offerings.",
  "industry": "Artificial Intelligence / Software Development / IT Consulting"
}


### Example 3 — REST tools

REST servers are registered with a base URL. Individual endpoints are registered as tools with their HTTP method, input/output JSON schemas, and a description.

This example uses the free [Open-Meteo](https://open-meteo.com/) weather API — no API key required.

In [24]:
kernel_rest = FunctionKernel()

# Register the Open-Meteo REST server
kernel_rest.register_rest_server(
    RestServer(name="weather", url="https://api.open-meteo.com/v1")
)

# Register the /forecast endpoint as a tool
kernel_rest.register_rest_tool(
    server_name="weather",
    tool_name="forecast",
    method="GET",
    input_schema={
        "type": "object",
        "properties": {
            "latitude": {"type": "number", "description": "Latitude of the location."},
            "longitude": {"type": "number", "description": "Longitude of the location."},
            "current": {
                "type": "string",
                "description": "Comma-separated list of current weather variables, e.g. temperature_2m,wind_speed_10m",
            },
        },
        "required": ["latitude", "longitude"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"},
            "current": {"type": "object"},
        },
    },
    description="Get current weather data for a GPS location from Open-Meteo.",
)

llm_client_rest = GeminiClient("gemini-2.5-flash")
agent_rest = Agent(llm_client=llm_client_rest, kernel=kernel_rest)

result_rest = await agent_rest.prompt(
    prompt="What is the current temperature in Tallinn, Estonia? Give brief answer.",
    max_steps=3,
)
print (result_rest)

2026-06-15 18:17:25.737 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 0/3
2026-06-15 18:17:27.879 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 696 tokens, 2.14s
2026-06-15 18:17:27.880 | INFO     | kavalai.agents.agent:_call_tool:241 - Calling tool rest://weather.forecast [GET] with {'latitude': 59.437, 'longitude': 24.7536, 'current': 'temperature_2m'}
2026-06-15 18:17:27.948 | INFO     | kavalai.functionkernel:_call_rest_tool:381 - Calling GET https://api.open-meteo.com/v1/forecast
2026-06-15 18:17:28.127 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 1/3
2026-06-15 18:17:29.610 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 795 tokens, 1.48s


The current temperature in Tallinn, Estonia is 16.1°C.


### Example 4 — MCP tools (stdio server)

MCP servers are registered with a shell command. The `FunctionKernel` starts the process, speaks the MCP JSON-RPC protocol over stdio, discovers available tools, and routes calls to them.

Here we spin up an in-process `FastMCP` server that exposes two arithmetic tools.

In [25]:
import sys
import textwrap

# Write a minimal FastMCP server to a temp file so the kernel can start it
mcp_server_path = "/tmp/kavalai_demo_mcp_server.py"
with open(mcp_server_path, "w") as f:
    f.write(textwrap.dedent("""
        from mcp.server.fastmcp import FastMCP

        mcp = FastMCP("demo-math")

        @mcp.tool()
        def add(a: float, b: float) -> float:
            \"\"\"Add two numbers.\"\"\"
            return a + b

        @mcp.tool()
        def multiply(a: float, b: float) -> float:
            \"\"\"Multiply two numbers.\"\"\"
            return a * b

        if __name__ == "__main__":
            mcp.run()
    """))

kernel_mcp = FunctionKernel()
kernel_mcp.register_mcp_server(
    McpServer(
        name="math",
        command=sys.executable,
        args=[mcp_server_path],
    )
)

llm_client_mcp = GeminiClient("gemini-2.5-flash")
agent_mcp = Agent(llm_client=llm_client_mcp, kernel=kernel_mcp)

result_mcp = await agent_mcp.prompt(
    prompt="What is (42 + 8) multiplied by 7?",
    max_steps=3,
)

print(result_mcp)
await kernel_mcp.close()

2026-06-15 18:17:38.827 | INFO     | kavalai.agents.agent:prompt:145 - Agent step 0/3
2026-06-15 18:17:39.896 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-2.5-flash): 327 tokens, 1.07s


350
